### get `featureClass.P_glcm`
by H.Nishiyama with GitHub Copilot<br>
ref: https://groups.google.com/g/pyradiomics/c/s1tbbvPFgyk<br>
2026-06-25 22:18:15

In [1]:
import os
import numpy as np
from radiomics import featureextractor
# from radiomics.glcm import RadiomicsGLCM
from glcm import RadiomicsGLCM
# from radiomics import base
# import six
import SimpleITK as sitk


In [2]:
dataDir = './'

imageName = sitk.ReadImage(os.path.join(dataDir, 'forRadiomicsTest.nrrd'))
maskName = sitk.ReadImage(os.path.join(dataDir, 'Segmentation.seg.nrrd'))

In [ ]:
# calculate parameter for GLCM
extractor = featureextractor.RadiomicsFeatureExtractor()
extractor.settings['distances'] = [1]
extractor.settings['symmetricalGLCM'] = True
#--- added
extractor.settings['weightingNorm'] = 'no_weighting'
# extractor.settings['weightingNorm'] = None  # default

#
extractor.settings['normalize'] = True
extractor.settings['label'] = 1  # target area's label
extractor.settings['binWidth'] = 1
extractor.settings['force2D'] = True # force 2D calculation


In [13]:

glcm_calc = RadiomicsGLCM(imageName, maskName, **extractor.settings)


In [14]:
# Check the GLCM coefficients
print("Ng (Num_grayLevels):", glcm_calc.coefficients['Ng'])
print("GrayLevels:", glcm_calc.coefficients['grayLevels'])


Ng (Num_grayLevels): 4
GrayLevels: [1 2 3 4]


In [15]:
glcm_calc.execute()  # calc GLCM

weigthing norm "True" is unknown, W is set to 1
weigthing norm "True" is unknown, W is set to 1
weigthing norm "True" is unknown, W is set to 1
weigthing norm "True" is unknown, W is set to 1
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


{'Autocorrelation': array(4.10952381),
 'ClusterProminence': array(12.53755185),
 'ClusterShade': array(1.97483598),
 'ClusterTendency': array(2.0131746),
 'Contrast': array(1.84285714),
 'Correlation': array(0.0441691),
 'DifferenceAverage': array(1.03333333),
 'DifferenceEntropy': array(1.77947293),
 'DifferenceVariance': array(0.77507937),
 'Id': array(0.60277778),
 'Idm': array(0.56428571),
 'Idmn': array(0.90981513),
 'Idn': array(0.81854875),
 'Imc1': array(-0.01100464),
 'Imc2': array(0.19778314),
 'InverseVariance': array(0.47116402),
 'JointAverage': array(2.01666667),
 'JointEnergy': array(0.09910431),
 'JointEntropy': array(3.6061393),
 'MCC': array(0.12543529),
 'MaximumProbability': array(0.15714286),
 'SumAverage': array(4.03333333),
 'SumEntropy': array(2.44303408),
 'SumSquares': array(0.96400794)}

In [16]:
p_glcm = glcm_calc.P_glcm  # get GLCM matrix [i,j,angle] with normalization 
print("GLCM shape:", p_glcm.shape)
print("GLCM matrix:", p_glcm)

GLCM shape: (1, 4, 4, 1)
GLCM matrix: [[[[0.11904762]
   [0.14285714]
   [0.06190476]
   [0.03095238]]

  [[0.14285714]
   [0.15714286]
   [0.0452381 ]
   [0.04761905]]

  [[0.06190476]
   [0.0452381 ]
   [0.00952381]
   [0.01666667]]

  [[0.03095238]
   [0.04761905]
   [0.01666667]
   [0.02380952]]]]


In [17]:
# glcm: 計算済みの RadiomicsGLCM インスタンス
# glcm.P_glcm の形状: (Nv, Ng, Ng, angles)

sums = np.nansum(glcm_calc.P_glcm, axis=(1, 2))  # 形: (Nv, angles)

# 表示（最初の ROI の各角度）
print("各角度の合計（ROI 0）:", sums[0])

# 検証（許容誤差 tol）
tol = 1e-6
ok = np.isclose(sums, 1.0, rtol=tol, atol=tol) | np.isnan(sums)
if np.all(ok):
    print("全ての角度で正規化されています（許容誤差内）。")
else:
    bad = np.argwhere(~ok)
    print("正規化されていない要素 (ROI, angle):", bad)
    for roi, angle in bad:
        print(f"ROI {roi}, angle {angle}, sum = {sums[roi, angle]}")

各角度の合計（ROI 0）: [1.]
全ての角度で正規化されています（許容誤差内）。


In [18]:
print(glcm_calc.sumP_glcm)
print(glcm_calc.raw_sumP_glcm)

[[420.]]
[[420.]]


In [19]:
import numpy as np

print("symmetricalGLCM (instance):", getattr(glcm_calc, "symmetricalGLCM", None))
print("P_glcm shape:", glcm_calc.P_glcm.shape)

P = glcm_calc.P_glcm  # (Nv, Ng, Ng, angles)
# 対称性チェック（NaN を等価扱い）
is_sym = np.allclose(P, np.transpose(P, (0, 2, 1, 3)), equal_nan=True)
print("全体として対称か:", is_sym)

# 角度ごとの最大差を確認
diff = P - np.transpose(P, (0, 2, 1, 3))
max_diff_per_angle = np.nanmax(np.abs(diff), axis=(1, 2))
print("角度ごとの max abs diff (各 ROI x angle):", max_diff_per_angle)

# 正規化前後の合計や削除の影響も確認
print("raw_sumP_glcm:", getattr(glcm_calc, "raw_sumP_glcm", None))
print("sumP_glcm:", getattr(glcm_calc, "sumP_glcm", None))

# 設定確認（重み付けなど）
print("extractor.settings['weightingNorm']:", extractor.settings.get("weightingNorm"))
print("extractor.settings['distances']:", extractor.settings.get("distances"))

symmetricalGLCM (instance): True
P_glcm shape: (1, 4, 4, 1)
全体として対称か: True
角度ごとの max abs diff (各 ROI x angle): [[0.]]
raw_sumP_glcm: [[420.]]
sumP_glcm: [[420.]]
extractor.settings['weightingNorm']: True
extractor.settings['distances']: [1]
